# PAD Analytics Phase 1 Caching - Manual Testing Guide

This notebook will test the Phase 1 caching functionality step by step.
Run each cell in order to verify the caching system works correctly.

## 🎯 What We're Testing
- ✅ Basic module imports
- ✅ CacheManager functionality
- ✅ CachedDataset with real data
- ✅ Image caching (optional)
- ✅ Performance comparisons

---

## TEST 1: Basic Imports

First, let's test that all the caching modules can be imported correctly.

In [8]:
import sys
import os

# Add src to path for local testing
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

print("=" * 60)
print("TEST 1: Basic Imports")
print("=" * 60)

try:
    import pad_analytics as pad
    print("✅ pad_analytics imported successfully")
    
    # Test individual caching components
    from pad_analytics import CacheManager
    print("✅ CacheManager imported")
    
    from pad_analytics import CachedDataset  
    print("✅ CachedDataset imported")
    
    print(f"\n📦 Package version: {pad.__version__}")
    
    print("\n🎉 All imports successful!")
    
except Exception as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Make sure you're running this from the project root directory")

TEST 1: Basic Imports
✅ pad_analytics imported successfully
✅ CacheManager imported
✅ CachedDataset imported

📦 Package version: 0.2.2

🎉 All imports successful!


In [29]:
# Example 1: Cache a different dataset
print("Creating CachedDataset for another dataset...")
dataset = CachedDataset('Leiberman-Lab_ChemoPADNNtraining2024_Partial-Drug-Set_v1.0')  # Model 16's dataset

# Load and cache metadata
print("Loading dataset metadata...")
start_time = time.time()
metadata = dataset.load_dataset_metadata()
load_time = time.time() - start_time

print(f"✅ Dataset loaded: {len(metadata)} records in {load_time:.3f}s")
print(f"First 3 records:")
print(metadata.head(3)[['id', 'sample_name', 'sample_id']])

# Check cache coverage
coverage = dataset.get_cache_coverage()
print(f"Cache coverage: {coverage['estimated_coverage_percent']}%")

# Cache some images (optional)
print("Caching 5 images...")
stats = dataset.download_and_cache_images(max_images=5, max_workers=2)
print(f"Cached {stats['cached_new']} new images, {stats['already_cached']} already cached")

# Example 2: Cache using model ID (automatically gets dataset)
from pad_analytics import padanalytics as pad

# Get dataset name from model ID
dataset_name = pad.get_dataset_name_from_model_id(18)  # PLS model
print(f"Dataset for model 18: {dataset_name}")

# Cache this dataset
cached_dataset = CachedDataset(dataset_name)
metadata = cached_dataset.load_dataset_metadata()
print(f"Loaded {len(metadata)} records")

# Example 3: Batch cache multiple datasets
datasets_to_cache = [
    "FHI2020_Stratified_Sampling",
    "ChemoPAD NN training 2024",
    "FHI2020_Stratified_Sampling_v2"
]

for dataset_name in datasets_to_cache:
    print(f"\nCaching {dataset_name}...")
    try:
        dataset = CachedDataset(dataset_name)
        metadata = dataset.load_dataset_metadata()
        print(f"✅ {len(metadata)} records cached")

        # Cache a few images from each
        stats = dataset.download_and_cache_images(max_images=3)
        print(f"Images: {stats['cached_new']} new, {stats['already_cached']} cached")

    except Exception as e:
        print(f"❌ Failed: {e}")

Creating CachedDataset for another dataset...
Loading dataset metadata...
Loading dataset: Leiberman-Lab_ChemoPADNNtraining2024_Partial-Drug-Set_v1.0
📡 Fetching dataset from PAD API...
✅ Dataset cached: Leiberman-Lab_ChemoPADNNtraining2024_Partial-Drug-Set_v1.0 (3609 records)
✅ Dataset loaded: 3609 records in 0.633s
First 3 records:
      id  sample_name  sample_id
0  47986  hydroxyurea      80903
1  47987  hydroxyurea      81560
2  47988  hydroxyurea      81560
Cache coverage: 0.0%
Caching 5 images...
🚀 Starting image caching for 5 images from Leiberman-Lab_ChemoPADNNtraining2024_Partial-Drug-Set_v1.0
Using 2 parallel workers
✅ Image cached successfully: 48fc8c211498d360dc01ca7f245754da
✅ Image cached successfully: 6624f905f8d350c899673bda764c102f
✅ Image cached successfully: 61ba6f73ec83e152a84ff68225d36391
✅ Image cached successfully: 533ad4e10b72291de3e1ad98dd99b030
✅ Image cached successfully: c064e2894c7c1b007e6adef18ddf2005
Progress: 5/5 (100.0%) - Elapsed: 0.3s

✅ Image caching

No dataset URLs found for: ChemoPAD NN training 2024
No dataset URLs found for: FHI2020_Stratified_Sampling_v2


❌ Failed: Failed to load dataset ChemoPAD NN training 2024: Dataset ChemoPAD NN training 2024 not found or empty

Caching FHI2020_Stratified_Sampling_v2...
Loading dataset: FHI2020_Stratified_Sampling_v2
📡 Fetching dataset from PAD API...
❌ Failed: Failed to load dataset FHI2020_Stratified_Sampling_v2: Dataset FHI2020_Stratified_Sampling_v2 not found or empty


## TEST 2: CacheManager Functionality

Test the core cache management capabilities.

In [9]:
print("=" * 60)
print("TEST 2: CacheManager Functionality")
print("=" * 60)

try:
    from pad_analytics import CacheManager
    
    # Create cache manager
    cache_mgr = CacheManager()
    print(f"✅ CacheManager created")
    print(f"   📁 Cache directory: {cache_mgr.cache_dir}")
    
    # Check directory structure
    print(f"   📂 Raw images: {cache_mgr.raw_images_dir}")
    print(f"   📂 Metadata: {cache_mgr.metadata_dir}")
    print(f"   📂 Datasets: {cache_mgr.datasets_dir}")
    
    # Get cache stats
    stats = cache_mgr.get_cache_stats()
    print(f"\n📊 Cache Statistics:")
    print(f"   • Total size: {stats['total_size_mb']} MB")
    print(f"   • Images: {stats['num_images']}")
    print(f"   • Datasets: {stats['num_datasets']}")
    
    # Test cache size calculation
    cache_size_bytes = cache_mgr._get_cache_size_bytes()
    cache_size_mb = cache_size_bytes / (1024 * 1024)
    print(f"   • Detailed size: {cache_size_mb:.2f} MB ({cache_size_bytes} bytes)")
    
    print("\n🎉 CacheManager test successful!")
    
except Exception as e:
    print(f"❌ CacheManager test failed: {e}")
    print("\n💡 This might be a permissions issue with the cache directory")

TEST 2: CacheManager Functionality
✅ CacheManager created
   📁 Cache directory: /home/pmoreira/.pad_cache
   📂 Raw images: /home/pmoreira/.pad_cache/raw_images
   📂 Metadata: /home/pmoreira/.pad_cache/metadata
   📂 Datasets: /home/pmoreira/.pad_cache/datasets

📊 Cache Statistics:
   • Total size: 8.84 MB
   • Images: 5
   • Datasets: 1
   • Detailed size: 8.84 MB (9272713 bytes)

🎉 CacheManager test successful!


## TEST 3: CachedDataset with Real Data

Test the cached dataset functionality using real PAD data.

In [10]:
print("=" * 60)
print("TEST 3: CachedDataset with Real Data")
print("=" * 60)

try:
    from pad_analytics import CachedDataset
    import time
    
    # Create cached dataset - using a known working dataset
    print("Creating CachedDataset for 'FHI2020_Stratified_Sampling'...")
    dataset = CachedDataset("FHI2020_Stratified_Sampling")
    
    # Load metadata (this tests both cache and API fallback)
    print("Loading dataset metadata...")
    start_time = time.time()
    metadata = dataset.load_dataset_metadata()
    load_time = time.time() - start_time
    
    print(f"✅ Dataset loaded: {len(metadata)} records in {load_time:.3f}s")
    print(f"   📋 Dataset representation: {dataset}")
    
    # Show first few rows
    print(f"\n📄 First 3 records:")
    print(metadata.head(3)[['id', 'sample_name', 'sample_id']].to_string())
    
    # Check cache coverage
    print("\nChecking cache coverage...")
    coverage = dataset.get_cache_coverage()
    print(f"   📊 Coverage: {coverage['estimated_coverage_percent']}%")
    print(f"   🔍 Sample: {coverage['sample_cached']}/{coverage['sample_size']} cached")
    print(f"   📈 Total cards: {coverage['total_cards']}")
    
    print(f"\n🎉 CachedDataset test successful!")
    
    # Store dataset for next tests
    globals()['test_dataset'] = dataset
    
except Exception as e:
    print(f"❌ CachedDataset test failed: {e}")
    print("\n💡 This might be due to network issues or API problems")
    globals()['test_dataset'] = None

TEST 3: CachedDataset with Real Data
Creating CachedDataset for 'FHI2020_Stratified_Sampling'...
Loading dataset metadata...
Loading dataset: FHI2020_Stratified_Sampling
✅ Loaded dataset from cache (8001 records)
✅ Dataset loaded: 8001 records in 0.035s
   📋 Dataset representation: CachedDataset('FHI2020_Stratified_Sampling', 8001 cards, ~0.0% cached)

📄 First 3 records:
      id         sample_name  sample_id
0  15589  hydroxychloroquine      53787
1  15590  hydroxychloroquine      53778
2  15591  hydroxychloroquine      53789

Checking cache coverage...
   📊 Coverage: 0.0%
   🔍 Sample: 0/100 cached
   📈 Total cards: 8001

🎉 CachedDataset test successful!


## TEST 4: Performance Comparison

Compare performance between cached and fresh dataset loading.

In [11]:
print("=" * 60)
print("TEST 4: Performance Comparison")
print("=" * 60)

if 'test_dataset' not in globals() or globals()['test_dataset'] is None:
    print("❌ No dataset available for performance test")
    print("   Please make sure TEST 3 completed successfully")
    print("   Trying to create a fresh dataset for comparison...")
    
    # Fallback: create a fresh dataset for testing
    try:
        from pad_analytics import CachedDataset
        import time
        
        # Test fresh dataset load
        print("Testing fresh dataset creation...")
        start_time = time.time()
        fresh_dataset = CachedDataset("FHI2020_Stratified_Sampling")
        fresh_load_time = time.time() - start_time
        
        # Test cached dataset load  
        print("Testing cached dataset load...")
        start_time = time.time()
        cached_data = fresh_dataset.load_dataset_metadata()
        cached_load_time = time.time() - start_time
        
        print(f"\n⏱️  Performance Results:")
        print(f"   • Fresh dataset: {fresh_load_time:.3f}s")
        print(f"   • Cached load: {cached_load_time:.3f}s")
        
        if cached_load_time < fresh_load_time:
            speedup = fresh_load_time / cached_load_time
            print(f"   🚀 Cache speedup: {speedup:.1f}x faster!")
        else:
            print(f"   📝 Similar performance (cache already optimized)")
            
        print("\n🎉 Performance test successful (fallback mode)!")
        
        # Store for next tests
        globals()['test_dataset'] = fresh_dataset
        
    except Exception as e:
        print(f"❌ Performance test failed: {e}")
        
else:
    try:
        import time
        from pad_analytics import CachedDataset
        
        dataset = globals()['test_dataset']
        
        # Test fresh dataset load
        print("Testing fresh dataset creation...")
        start_time = time.time()
        fresh_dataset = CachedDataset("FHI2020_Stratified_Sampling")
        fresh_load_time = time.time() - start_time
        
        # Test cached dataset load  
        print("Testing cached dataset load...")
        start_time = time.time()
        cached_data = dataset.load_dataset_metadata()
        cached_load_time = time.time() - start_time
        
        print(f"\n⏱️  Performance Results:")
        print(f"   • Fresh dataset: {fresh_load_time:.3f}s")
        print(f"   • Cached load: {cached_load_time:.3f}s")
        
        if cached_load_time < fresh_load_time:
            speedup = fresh_load_time / cached_load_time
            print(f"   🚀 Cache speedup: {speedup:.1f}x faster!")
        else:
            print(f"   📝 Similar performance (cache already optimized)")
            
        print("\n🎉 Performance test successful!")
            
    except Exception as e:
        print(f"❌ Performance test failed: {e}")

TEST 4: Performance Comparison
Testing fresh dataset creation...
Testing cached dataset load...
Loading dataset: FHI2020_Stratified_Sampling
✅ Loaded dataset from cache (8001 records)

⏱️  Performance Results:
   • Fresh dataset: 0.001s
   • Cached load: 0.030s
   📝 Similar performance (cache already optimized)

🎉 Performance test successful!


## TEST 5: Image Caching (Optional)

⚠️ **Warning**: This test downloads actual images and requires internet connection.
It may take several minutes to complete. Only run if you want to test the complete workflow.

**Skip this test if you want a quick verification.**

Loading dataset: FHI2020_Stratified_Sampling
✅ Loaded dataset from cache (8001 records)


,id,sample_id,sample_name,quantity,camera_type_1,url,hashlib_md5,image_name
0,15589,53787,hydroxychloroquine,100,Google Pixel 3a,https://pad.crc.nd.edu//var/www/html/images/pa...,c7ffc09ba273d13dfd9f295aa2f66cb5,15589__53787__hydroxychloroquine__100.png
1,15590,53778,hydroxychloroquine,100,Google Pixel 3a,https://pad.crc.nd.edu//var/www/html/images/pa...,ac2f4e2656289c2c26c087bc3b918ed5,15590__53778__hydroxychloroquine__100.png
2,15591,53789,hydroxychloroquine,100,Google Pixel 3a,https://pad.crc.nd.edu//var/www/html/images/pa...,7f983c5681156fd10cd49c592ea3d25e,15591__53789__hydroxychloroquine__100.png
3,15592,53787,hydroxychloroquine,100,Google Pixel 3a,https://pad.crc.nd.edu//var/www/html/images/pa...,7c5e34b986e53ec0e05b0dfe086b4c2e,15592__53787__hydroxychloroquine__100.png
4,15595,53785,hydroxychloroquine,100,Google Pixel 3a,https://pad.crc.nd.edu//var/www/html/images/pa...,a765e98e08c00e506b511b39cdef0ac9,15595__53785__hydroxychloroquine__100.png
...,...,...,...,...,...,...,...,...
7996,44892,65509,swiped-but-not-run,0,iPhone,https://pad.crc.nd.edu//var/www/html/images/pa...,ae8f209d31be8cf9ff6145dac491d5a2,44892__65509__swiped-but-not-run__0.png
7997,44895,65509,swiped-but-not-run,0,Google Pixel 3a,https://pad.crc.nd.edu//var/www/html/images/pa...,28a4aa41d314812d36fc608810d60630,44895__65509__swiped-but-not-run__0.png
7998,44898,65322,swiped-but-not-run,0,Google Pixel 3a,https://pad.crc.nd.edu//var/www/html/images/pa...,1b7822dabdb7e590a96b0d0efcb52a16,44898__65322__swiped-but-not-run__0.png
7999,44899,65404,swiped-but-not-run,0,Google Pixel 3a,https://pad.crc.nd.edu//var/www/html/images/pa...,04d74c3d14d87310880d8726cff60719,44899__65404__swiped-but-not-run__0.png


In [27]:
# Set this to True if you want to test image caching
TEST_IMAGE_CACHING = True  # Change to True to enable
NUM_TEST_IMAGES = 3  # Small number for testing

print("=" * 60)
print(f"TEST 5: Image Caching ({NUM_TEST_IMAGES} images)")
print("=" * 60)

if not TEST_IMAGE_CACHING:
    print("⏭️  Image caching test DISABLED")
    print("   Set TEST_IMAGE_CACHING = True above to enable this test")
    print("   This test requires internet connection and takes several minutes")
elif 'test_dataset' not in globals() or globals()['test_dataset'] is None:
    print("❌ No dataset available for image caching test")
    print("   Please make sure TEST 3 completed successfully")
    print("   Trying to create a dataset for image caching test...")
    
    # Fallback: create a fresh dataset for image caching
    try:
        from pad_analytics import CachedDataset
        print("Creating fresh dataset for image caching...")
        dataset = CachedDataset("FHI2020_Stratified_Sampling")
        dataset.load_dataset_metadata()
        
        print(f"Attempting to cache {NUM_TEST_IMAGES} images...")
        print("⚠️  This requires internet connection and may take a moment...")
        print("⏳ Please wait...")
        
        # Download and cache a small number of images
        stats = dataset.download_and_cache_images(
            max_images=NUM_TEST_IMAGES,
            max_workers=2  # Conservative for testing
        )
        
        print(f"\n✅ Image caching completed!")
        print(f"   📊 Results:")
        print(f"   • Total images: {stats['total_images']}")
        print(f"   • Newly cached: {stats['cached_new']}")
        print(f"   • Already cached: {stats['already_cached']}")
        print(f"   • Failed: {stats['failed']}")
        print(f"   • Total time: {stats['total_time']:.1f}s")
        
        if stats['cached_new'] > 0:
            avg_time = stats['total_time'] / stats['cached_new']
            print(f"   • Avg time per image: {avg_time:.2f}s")
        
        print("\n🎉 Image caching test successful (fallback mode)!")
        
    except Exception as e:
        print(f"❌ Image caching failed: {e}")
        print(f"   💡 This might be due to network issues or missing dependencies")
        print(f"   💡 The core caching system can still work without image downloads")
        
else:
    try:
        dataset = globals()['test_dataset']
        
        print(f"Attempting to cache {NUM_TEST_IMAGES} images...")
        print("⚠️  This requires internet connection and may take a moment...")
        print("⏳ Please wait...")
        
        # Download and cache a small number of images
        stats = dataset.download_and_cache_images(
            max_images=NUM_TEST_IMAGES,
            max_workers=2  # Conservative for testing
        )
        
        print(f"\n✅ Image caching completed!")
        print(f"   📊 Results:")
        print(f"   • Total images: {stats['total_images']}")
        print(f"   • Newly cached: {stats['cached_new']}")
        print(f"   • Already cached: {stats['already_cached']}")
        print(f"   • Failed: {stats['failed']}")
        print(f"   • Total time: {stats['total_time']:.1f}s")
        
        if stats['cached_new'] > 0:
            avg_time = stats['total_time'] / stats['cached_new']
            print(f"   • Avg time per image: {avg_time:.2f}s")
        
        print("\n🎉 Image caching test successful!")
        
    except Exception as e:
        print(f"❌ Image caching failed: {e}")
        print(f"   💡 This might be due to network issues or missing dependencies")
        print(f"   💡 The core caching system can still work without image downloads")

TEST 5: Image Caching (3 images)
Attempting to cache 3 images...
⚠️  This requires internet connection and may take a moment...
⏳ Please wait...
❌ Image caching failed: 'str' object has no attribute 'download_and_cache_images'
   💡 This might be due to network issues or missing dependencies
   💡 The core caching system can still work without image downloads


In [25]:
test_dataset


'FHI2020_Stratified_Sampling'

In [26]:
# Set this to True if you want to test image caching
TEST_IMAGE_CACHING = True  # Change to True to enable
NUM_TEST_IMAGES = 3  # Small number for testing

print("=" * 60)
print(f"TEST 5: Image Caching ({NUM_TEST_IMAGES} images)")
print("=" * 60)

test_dataset = pad.get_dataset_name_from_model_id(16) #         "ChemoPAD NN training 2024"

if not TEST_IMAGE_CACHING:
    print("⏭️  Image caching test DISABLED")
    print("   Set TEST_IMAGE_CACHING = True above to enable this test")
    print("   This test requires internet connection and takes several minutes")
elif 'test_dataset' not in globals() or globals()['test_dataset'] is None:
    print("❌ No dataset available for image caching test")
    print("   Please make sure TEST 3 completed successfully")
else:
    try:
        
        dataset = globals()['test_dataset']
        
        print(f"Attempting to cache {NUM_TEST_IMAGES} images...")
        print("⚠️  This requires internet connection and may take a moment...")
        print("⏳ Please wait...")
        
        # Download and cache a small number of images
        stats = dataset.download_and_cache_images(
            max_images=NUM_TEST_IMAGES,
            max_workers=2  # Conservative for testing
        )
        
        print(f"\n✅ Image caching completed!")
        print(f"   📊 Results:")
        print(f"   • Total images: {stats['total_images']}")
        print(f"   • Newly cached: {stats['cached_new']}")
        print(f"   • Already cached: {stats['already_cached']}")
        print(f"   • Failed: {stats['failed']}")
        print(f"   • Total time: {stats['total_time']:.1f}s")
        
        if stats['cached_new'] > 0:
            avg_time = stats['total_time'] / stats['cached_new']
            print(f"   • Avg time per image: {avg_time:.2f}s")
        
        print("\n🎉 Image caching test successful!")
        
    except Exception as e:
        print(f"❌ Image caching failed: {e}")
        print(f"   💡 This might be due to network issues or missing dependencies")
        print(f"   💡 The core caching system can still work without image downloads")

TEST 5: Image Caching (3 images)
Attempting to cache 3 images...
⚠️  This requires internet connection and may take a moment...
⏳ Please wait...
❌ Image caching failed: 'str' object has no attribute 'download_and_cache_images'
   💡 This might be due to network issues or missing dependencies
   💡 The core caching system can still work without image downloads


## TEST 6: Final Verification

Run a final check to verify all caching functionality is working.

In [15]:
print("=" * 60)
print("TEST 6: Final Verification & Summary")
print("=" * 60)

try:
    # Get final cache statistics
    cache_mgr = CacheManager()
    final_stats = cache_mgr.get_cache_stats()
    
    # Get dataset status if available
    if 'test_dataset' in globals() and globals()['test_dataset'] is not None:
        dataset = globals()['test_dataset']
        final_coverage = dataset.get_cache_coverage()
        offline_ready = dataset.is_offline_ready()
    else:
        final_coverage = None
        offline_ready = False
    
    print("📊 Final Cache Status:")
    print(f"   • Cache directory: {final_stats['cache_dir']}")
    print(f"   • Total size: {final_stats['total_size_mb']} MB")
    print(f"   • Cached images: {final_stats['num_images']}")
    print(f"   • Cached datasets: {final_stats['num_datasets']}")
    
    if final_coverage:
        print(f"   • Dataset coverage: {final_coverage['estimated_coverage_percent']}%")
        print(f"   • Ready for offline use: {offline_ready}")
    
    # Check cache directory exists
    import os
    cache_exists = os.path.exists(cache_mgr.cache_dir)
    print(f"   • Cache directory exists: {cache_exists}")
    
    if cache_exists:
        subdirs = ['raw_images', 'metadata', 'datasets']
        for subdir in subdirs:
            subdir_path = cache_mgr.cache_dir / subdir
            exists = subdir_path.exists()
            print(f"   • {subdir}/ exists: {exists}")
    
    print("\n" + "=" * 60)
    print("🎉 Phase 1 Testing Complete!")
    print("=" * 60)
    
    print("\n📝 Summary:")
    print("   ✅ Core caching infrastructure working")
    print("   ✅ Dataset metadata caching functional") 
    print("   ✅ Cache management utilities operational")
    print("   ✅ Ready for Phase 2 development")
    
    print(f"\n💡 Tips for further exploration:")
    print(f"   • Try: dataset.get_cache_coverage()")
    print(f"   • Try: cache_mgr.get_cache_stats()") 
    print(f"   • Try: dataset.download_and_cache_images(max_images=10)")
    print(f"   • Explore cache directory: ~/.pad_cache/")
    
except Exception as e:
    print(f"❌ Final verification failed: {e}")
    print("   Some components may not be working correctly")

TEST 6: Final Verification & Summary
❌ Final verification failed: 'str' object has no attribute 'get_cache_coverage'
   Some components may not be working correctly


## 🎯 Next Steps

If all tests passed successfully, the Phase 1 caching system is working correctly!

### ✅ What Should Work:
- **Imports**: All caching modules load without critical errors
- **Cache Manager**: Creates directory structure and reports accurate stats
- **Dataset Caching**: Loads FHI2020_Stratified_Sampling quickly from cache
- **Performance**: Cache loading is much faster than fresh API calls
- **Image Caching**: (If tested) Downloads and stores images successfully

### ⚠️ Expected Warnings:
- Warnings about `ipywidgets` or `tensorflow` are **normal** - the caching system works independently
- SSL certificate warnings are **normal** - we use `verify=False` for the PAD API

### 🚀 Ready for Phase 2!
If the tests completed successfully, we can proceed to implement:
- **Phase 2**: Preprocessing Pipeline Abstraction
- **Phase 3**: Model Adapter Interface  
- **Phase 4**: Advanced Features

---

**Questions or Issues?** 
- Check the cache directory: `~/.pad_cache/`
- Verify internet connection for API calls
- Make sure you're running from the project root directory